Work off the CSV first (authoritative set)
- Iterate only the columns you actually use (e.g., judgement, order, seizure, opinion, third-party, ruling, view).
- For each cell that contains a relative HTML path, open that file, parse the title, and check if it’s exactly the placeholder title.
- If it is, move that file into a quarantine subfolder, and clear the CSV cell.
- Optionally, if a CSV cell points to a non-existent file, you can choose to clear it too (flagged via a switch).

Then (optional) sweep the entire HTML directory:
- Walk the HTML tree and quarantine any placeholder files not referenced in the CSV. This catches orphans your CSV doesn’t know about.
- Make it idempotent, auditable, and safe
- Dry-run mode first (no changes), then run for real.
- Automatic CSV backup (.bak) and atomic write.
- Preserve relative paths when moving, and avoid overwriting by adding numeric suffixes if a file already exists in quarantine.
- Minimal parsing: extract <title> and compare case-insensitively.

In [1]:
import os
import re
import time
import shutil
import hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from html.parser import HTMLParser


In [2]:
def ensure_dir(path):
    if not os.path.exists(path):
        print(f"{path} does not exist. Creating it...")
        os.makedirs(path)

In [3]:
# Base directory containing your saved HTML files (recurses subfolders)
BASE_DIR = os.getcwd()
print(BASE_DIR)

C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper


In [4]:
sources = ["c1_juris", "c2_juris", "t2_juris"]

In [5]:
HTML_COLUMNS = ["judgement", "order", "seizure", "opinion", "third-party", "ruling", "view"]

In [6]:
all_html_files = {}

In [7]:
PLACEHOLDER_TITLE = "EUR-Lex - Official Journal of the European Union"

class TitleParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self._in_title = False
        self.title_text = ""

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "title":
            self._in_title = True

    def handle_endtag(self, tag):
        if tag.lower() == "title":
            self._in_title = False

    def handle_data(self, data):
        if self._in_title:
            self.title_text += data

def extract_title_from_file(path: str, max_bytes: int = 2_000_000):
    try:
        with open(path, "rb") as f:
            chunk = f.read(max_bytes)
        text = chunk.decode("utf-8", errors="ignore")
        p = TitleParser()
        p.feed(text)
        t = p.title_text.strip()
        return t if t else None
    except Exception:
        return None

def is_placeholder_html(path: str) -> bool:
    t = extract_title_from_file(path)
    return (t is not None) and (t.strip().lower() == PLACEHOLDER_TITLE.lower())

In [8]:
to_quarantine = {}

for src in sources:
    src_path = os.path.join(BASE_DIR, src)
    csv_path = os.path.join(src_path, f"{src}.csv")
    dest_subdir = os.path.join(src_path, "official_journal")
    ensure_dir(dest_subdir)

    if not os.path.exists(csv_path):
        print(f"⚠️ CSV not found for {src}: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    print(f"✅ Opened {csv_path} with {len(df)} rows")

    html_files = set()
    empty_count = 0
    checked = 0
    placeholders = set()

    # go through the relevant columns, if they exist
    for col in HTML_COLUMNS:
        if col not in df.columns:
            continue
                # drop NaNs, strip whitespace
        values = df[col].dropna().astype(str).str.strip()
        for val in values:
            if val.endswith(".html"):
                html_files.add(val)  # keep source prefix for later file opening
                val_norm = val.replace("\\", "/")
                abs_path = os.path.normpath(os.path.join(BASE_DIR, val_norm))
                checked += 1
                if is_placeholder_html(abs_path):
                    empty_count += 1
                    placeholders.add(val_norm)  # NEW: add to quarantine list
                    print(f"EMPTY PAGE: {os.path.relpath(abs_path, BASE_DIR)}")

    all_html_files[src] = html_files
    to_quarantine[src] = placeholders  # NEW: store detected placeholders
    print(f"  → found {len(html_files)} unique HTML files referenced in {src}")
    print(f"  → Checked {checked} files in {src}; placeholder pages: {empty_count}")

print("\nSummary:")
for k, v in all_html_files.items():
    print(f"{k}: {len(v)} files")

print("\n=== To Quarantine Summary ===")
total_quarantine = 0
for src, files in to_quarantine.items():
    print(f"{src}: {len(files)} placeholder files")
    total_quarantine += len(files)

print(f"\nTOTAL placeholder files across all sources: {total_quarantine}")


✅ Opened C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\c1_juris\c1_juris.csv with 5653 rows
EMPTY PAGE: c1_juris\61970CO0080.html
EMPTY PAGE: c1_juris\61973CO0075.html
EMPTY PAGE: c1_juris\61973CO0076.html
EMPTY PAGE: c1_juris\61974CO0053.html
EMPTY PAGE: c1_juris\61974CO0055.html
EMPTY PAGE: c1_juris\61974CO0057.html
EMPTY PAGE: c1_juris\61975CO0025.html
EMPTY PAGE: c1_juris\61975CO0027.html
EMPTY PAGE: c1_juris\61973CS0073.html
EMPTY PAGE: c1_juris\61973CS0077.html
EMPTY PAGE: c1_juris\61973CS0078.html
EMPTY PAGE: c1_juris\61973CS0081.html
EMPTY PAGE: c1_juris\61974CS0053.html
EMPTY PAGE: c1_juris\61974CS0055.html
EMPTY PAGE: c1_juris\61974CS0056.html
EMPTY PAGE: c1_juris\61975CS0024.html
EMPTY PAGE: c1_juris\61975CS0025.html
EMPTY PAGE: c1_juris\61958CV0036.html
EMPTY PAGE: c1_juris\61970CV0078.html
EMPTY PAGE: c1_juris\61973CV0075.html
EMPTY PAGE: c1_juris\61973CV0076.html
EMPTY PAGE: c1_juris\61974CV0054.html
EMPTY PAGE: c1_juris\61958CT0033.html
EMPTY PAGE:

In [15]:
for src, files in to_quarantine.items():
    src_path = os.path.join(BASE_DIR, src)
    dest_path = os.path.join(src_path, "official_journal")
    os.makedirs(dest_path, exist_ok=True)

    # Deduplicate file names
    unique_files = set(files)

    for file_name in unique_files:
        # --- FIX: make sure file path is correct regardless of prefix ---
        file_name_norm = file_name.replace("\\", "/")
        if file_name_norm.startswith(src + "/") or file_name_norm.startswith(src + "\\"):
            # file_name already contains the src folder → make absolute from BASE_DIR
            src_file = os.path.join(BASE_DIR, file_name_norm)
        else:
            # file_name is relative to the src folder
            src_file = os.path.join(src_path, file_name_norm)
        src_file = os.path.normpath(src_file)
        # ----------------------------------------------------------------

        if not os.path.exists(src_file):
            continue  # skip missing files silently

        dest_file = os.path.join(dest_path, os.path.basename(file_name_norm))

        # Avoid overwriting: add numeric suffix if file already exists
        base, ext = os.path.splitext(dest_file)
        counter = 1
        while os.path.exists(dest_file):
            dest_file = f"{base}_{counter}{ext}"
            counter += 1

        shutil.move(src_file, dest_file)

    print(f"✅ {src}: moved {len(unique_files)} unique files → {dest_path}")

print("\n🎉 All quarantine files moved and deduplicated.")

✅ c1_juris: moved 41 unique files → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\c1_juris\official_journal
✅ c2_juris: moved 2 unique files → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\c2_juris\official_journal
✅ t2_juris: moved 4564 unique files → C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\t2_juris\official_journal

🎉 All quarantine files moved and deduplicated.


In [14]:
# %% [markdown]
# ### Replace quarantined HTML filenames with "X" and save to new CSVs prefixed with "n_"

# %%
import pandas as pd
import os

for src, files in to_quarantine.items():
    src_path = os.path.join(BASE_DIR, src)
    csv_path = os.path.join(src_path, f"{src}.csv")
    new_csv_path = os.path.join(src_path, f"n_{src}.csv")

    if not os.path.exists(csv_path):
        print(f"⚠️ No CSV found for {src}, skipping.")
        continue

    df = pd.read_csv(csv_path)
    print(f"🧾 Opened {csv_path} with {len(df)} rows")

    basenames = {os.path.basename(f) for f in set(files)}

    # We'll use a mutable object (list) to track updates
    updated = [0]

    for col in HTML_COLUMNS:
        if col not in df.columns:
            continue

        def replace_if_quarantined(val):
            if isinstance(val, str) and val.endswith(".html"):
                for b in basenames:
                    if b in val:
                        updated[0] += 1
                        return "X"
            return val

        df[col] = df[col].apply(replace_if_quarantined)

    df.to_csv(new_csv_path, index=False)
    print(f"✅ {src}: replaced {updated[0]} quarantined references with 'X' → saved as {os.path.basename(new_csv_path)}")

print("\n🎉 All quarantined HTML names replaced and saved to new CSVs prefixed with 'n_'.")



🧾 Opened C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\c1_juris\c1_juris.csv with 5653 rows
✅ c1_juris: replaced 41 quarantined references with 'X' → saved as n_c1_juris.csv
🧾 Opened C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\c2_juris\c2_juris.csv with 22269 rows
✅ c2_juris: replaced 2 quarantined references with 'X' → saved as n_c2_juris.csv
🧾 Opened C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper\t2_juris\t2_juris.csv with 22581 rows
✅ t2_juris: replaced 4857 quarantined references with 'X' → saved as n_t2_juris.csv

🎉 All quarantined HTML names replaced and saved to new CSVs prefixed with 'n_'.
